# LeetCode 1245 — Tree Diameter bằng Python

Link đề: https://leetcode.com/problems/tree-diameter/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Graph Theory, Topological Sort, DP on Trees

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề và chữ ký hàm được đối chiếu với https://leetcode.doocs.org/en/lc/1245/ (`class Solution`, method `treeDiameter(self, edges)` trả về `int`).

Skeleton LeetCode:

```python
class Solution:
    def treeDiameter(self, edges: List[List[int]]) -> int:
        ...
```

Ghi chú: đầu vào là **danh sách cạnh** của một cây vô hướng (không phải `TreeNode`). Đường kính tính bằng **số cạnh**, không phải số đỉnh.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Dựng **danh sách kề** (adjacency list) cho cây vô hướng từ danh sách cạnh, và hiểu vì sao cây có đúng `n − 1` cạnh, giữa hai đỉnh có **duy nhất** một đường đi.
2. Viết BFS tính khoảng cách từ một đỉnh tới mọi đỉnh; từ đó có lời giải vét cạn `O(n²)`.
3. Nắm thủ thuật **hai lần BFS** (double sweep): đỉnh xa nhất từ một đỉnh bất kỳ luôn là **một đầu mút của đường kính**.
4. Làm **DP trên cây** (tree DP): tại mỗi đỉnh lấy **hai nhánh sâu nhất** của các con; cài đặt **không đệ quy** để chịu được cây dạng đường thẳng `10⁴` đỉnh.
5. Biết cách **bóc lá theo lớp** (giống topological sort / bài 310) để suy ra đường kính.

## 2. Đề bài & ví dụ

**Đường kính** (diameter) của một cây là **số cạnh** trên đường đi dài nhất trong cây.

Có một cây vô hướng `n` đỉnh đánh số `0` đến `n − 1`. Cho mảng `edges` với `edges.length == n − 1`, `edges[i] = [aᵢ, bᵢ]` nghĩa là có cạnh vô hướng giữa `aᵢ` và `bᵢ`. Trả về đường kính của cây.

**Ví dụ 1:** `edges = [[0,1],[0,2]]` → `2` (đường dài nhất `1 − 0 − 2`).

**Ví dụ 2:** `edges = [[0,1],[1,2],[2,3],[1,4],[4,5]]` → `4` (đường `3 − 2 − 1 − 4 − 5`).

**Ràng buộc:** `n == edges.length + 1`; `1 <= n <= 10⁴`; `0 <= aᵢ, bᵢ < n`; `aᵢ != bᵢ`.

Lưu ý `n = 1` thì `edges = []` và đáp án là `0`.

In [1]:
import random
import sys
import time
from collections import deque, defaultdict
from typing import List

examples = [
    ([[0, 1], [0, 2]], 2),
    ([[0, 1], [1, 2], [2, 3], [1, 4], [4, 5]], 4),
    ([], 0),
]
for edges, want in examples:
    print(f"n = {len(edges) + 1}, edges = {edges} -> {want}")

n = 3, edges = [[0, 1], [0, 2]] -> 2
n = 6, edges = [[0, 1], [1, 2], [2, 3], [1, 4], [4, 5]] -> 4
n = 1, edges = [] -> 0


## 3. Kiến thức nền

### 3.1 Danh sách kề cho đồ thị vô hướng

Mỗi cạnh `[a, b]` thêm `b` vào hàng xóm của `a` **và** `a` vào hàng xóm của `b`. Số đỉnh `n = len(edges) + 1` (đừng suy từ `max` nhãn — sẽ hỏng khi `edges` rỗng).

In [2]:
def build_graph(edges):
    n = len(edges) + 1
    g = [[] for _ in range(n)]
    for a, b in edges:
        g[a].append(b)
        g[b].append(a)
    return g


g = build_graph(examples[1][0])
for v, nb in enumerate(g):
    print(v, "->", nb)

0 -> [1]
1 -> [0, 2, 4]
2 -> [1, 3]
3 -> [2]
4 -> [1, 5]
5 -> [4]


### 3.2 Tính chất của cây

- Liên thông, không có chu trình, `n` đỉnh thì `n − 1` cạnh.
- Giữa hai đỉnh bất kỳ có **đúng một** đường đi đơn → "khoảng cách" giữa hai đỉnh được xác định duy nhất.
- **Lá** (leaf) là đỉnh bậc 1. Cây có ≥ 2 đỉnh thì có ít nhất 2 lá.

Khi duyệt cây vô hướng, chỉ cần nhớ **cha** (parent) để không đi ngược lại — không cần mảng `visited` đầy đủ.

In [3]:
degree = [len(nb) for nb in g]
print("bậc:", degree, "| lá:", [v for v in range(len(g)) if degree[v] == 1])
print("số cạnh = n - 1 ?", len(examples[1][0]) == len(g) - 1)

bậc: [1, 3, 2, 1, 2, 1] | lá: [0, 3, 5]
số cạnh = n - 1 ? True


### 3.3 BFS tính khoảng cách

BFS (Breadth-First Search) từ đỉnh `s` thăm các đỉnh theo thứ tự khoảng cách tăng dần. Với cây (cạnh không trọng số), `dist[v]` thu được chính là số cạnh trên đường duy nhất từ `s` đến `v`.

In [4]:
def bfs_dist(g, s):
    dist = [-1] * len(g)
    dist[s] = 0
    q = deque([s])
    while q:
        u = q.popleft()
        for v in g[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist


print("dist từ 0:", bfs_dist(g, 0))
print("dist từ 3:", bfs_dist(g, 3))

dist từ 0: [0, 1, 2, 3, 2, 3]
dist từ 3: [3, 2, 1, 0, 3, 4]


### 3.4 Giới hạn đệ quy của Python

Cây có thể là một **đường thẳng** `0 − 1 − 2 − … − 9999`. DFS đệ quy sẽ sâu `10⁴` tầng, vượt giới hạn mặc định (~1000) → `RecursionError`. Vì vậy các cách dưới đây đều cài đặt **lặp** (dùng `deque`/stack, hoặc thứ tự BFS ngược).

In [5]:
print("recursion limit:", sys.getrecursionlimit())

recursion limit: 1000


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2. Vẽ cây:

```
        0
        |
  3 — 2 — 1 — 4 — 5
```

Liệt kê một vài khoảng cách: `d(3,5) = 4` (3-2-1-4-5), `d(0,5) = 3`, `d(0,3) = 3`, `d(2,4) = 2`. Lớn nhất là `4` → đường kính 4.

Quan sát quan trọng: bắt đầu BFS từ **đỉnh bất kỳ**, ví dụ `0`: các đỉnh xa nhất là `3` và `5` (cách 3). Chúng chính là hai đầu đường kính! BFS lần nữa từ `3`: đỉnh xa nhất là `5` với khoảng cách `4` = đường kính.

In [6]:
d0 = bfs_dist(g, 0)
far = max(range(len(g)), key=lambda v: d0[v])
print("từ 0, đỉnh xa nhất:", far, "khoảng cách", d0[far])
d1 = bfs_dist(g, far)
print("từ", far, "xa nhất:", max(d1), "-> đường kính")

từ 0, đỉnh xa nhất: 3 khoảng cách 3
từ 3 xa nhất: 4 -> đường kính


## 5. Cách 1 — Brute Force / baseline

Đường kính = `max` khoảng cách giữa mọi cặp đỉnh. Chạy BFS từ **mọi** đỉnh, lấy `max`.

In [7]:
class SolutionBrute:
    def treeDiameter(self, edges):
        g = build_graph(edges)
        return max(max(bfs_dist(g, s)) for s in range(len(g)))


for edges, want in examples:
    print(SolutionBrute().treeDiameter(edges), want)

2 2
4 4
0 0


**Độ phức tạp:** `n` lần BFS, mỗi lần `O(n)` → `O(n²)` thời gian = `10⁸` với `n = 10⁴` — quá chậm trong Python. Bộ nhớ `O(n)`.

**Nút thắt:** ta tính khoảng cách giữa **mọi** cặp, trong khi chỉ cần **một** cặp (hai đầu đường kính). Câu hỏi: tìm được một đầu mút đường kính mà không thử hết?

## 6. Bước đột phá

**Insight 1 — Double sweep:** từ đỉnh bất kỳ `s`, đỉnh `u` xa `s` nhất **luôn là một đầu mút của một đường kính nào đó**. Sau đó BFS từ `u`, khoảng cách xa nhất chính là đường kính. Chỉ cần **2 lần BFS** → `O(n)`.

**Insight 2 — Tree DP:** treo cây tại gốc `0`. Mọi đường đi đơn có một đỉnh "cao nhất" (gần gốc nhất) `v`; đường đó đi xuống **hai nhánh con khác nhau** của `v` (hoặc một nhánh). Nếu `down[v]` = độ sâu lớn nhất đi xuống từ `v` (tính bằng cạnh), thì đường dài nhất có đỉnh cao nhất là `v` bằng **tổng hai giá trị lớn nhất** trong `{down[c] + 1 : c là con của v}` (thiếu thì tính 0).

**Quy tắc:** "nếu xét tại đỉnh `v` thì đường kính ứng viên = `best1 + best2`, với `best1 ≥ best2` là hai nhánh sâu nhất; trả về cho cha giá trị `best1`."

**Insight 3 — Bóc lá:** liên tục xoá mọi lá (theo lớp). Mỗi lớp làm đường kính giảm đúng 2 (cả hai đầu mút đường kính đều là lá). Còn lại 1 đỉnh → đường kính `2 · lớp`; còn lại 2 đỉnh (một cạnh) → `2 · lớp + 1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai lần BFS (double sweep)

In [8]:
class SolutionTwoBFS:
    def treeDiameter(self, edges):
        g = build_graph(edges)

        def farthest(s):
            dist = bfs_dist(g, s)
            u = max(range(len(g)), key=dist.__getitem__)
            return u, dist[u]

        u, _ = farthest(0)
        _, d = farthest(u)
        return d


for edges, want in examples:
    print(SolutionTwoBFS().treeDiameter(edges), want)

2 2
4 4
0 0


**Độ phức tạp:** 2 lần BFS → `O(n)` thời gian, `O(n)` bộ nhớ. Không đệ quy.

### 7.2 Cách 3 — Tree DP lặp (hai nhánh sâu nhất)

Để xử lý con **trước** cha mà không đệ quy: BFS từ gốc lấy thứ tự `order` (cha luôn đứng trước con), rồi duyệt `order` **ngược** — khi đó mọi con đã được tính xong trước cha.

In [9]:
class SolutionTreeDP:
    def treeDiameter(self, edges):
        g = build_graph(edges)
        n = len(g)
        parent = [-1] * n
        order = [0]
        seen = [False] * n
        seen[0] = True
        for u in order:                      # BFS: list vừa duyệt vừa nối thêm
            for v in g[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    order.append(v)
        best1 = [0] * n                      # nhánh sâu nhất đi xuống từ v
        best2 = [0] * n                      # nhánh sâu thứ hai (con khác)
        ans = 0
        for v in reversed(order):
            ans = max(ans, best1[v] + best2[v])
            p = parent[v]
            if p >= 0:
                d = best1[v] + 1
                if d > best1[p]:
                    best1[p], best2[p] = d, best1[p]
                elif d > best2[p]:
                    best2[p] = d
        return ans


for edges, want in examples:
    print(SolutionTreeDP().treeDiameter(edges), want)

2 2
4 4
0 0


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ, **một** lần duyệt. Ưu điểm so với double sweep: tổng quát hoá được cho cây có **trọng số âm** hoặc khi cần thêm thông tin tại mỗi đỉnh.

### 7.3 Cách 4 — Bóc lá theo lớp (topological peeling)

Giống thuật toán tìm tâm cây (LeetCode 310). Dùng mảng bậc; mỗi vòng xoá toàn bộ lá hiện tại.

In [10]:
class SolutionPeel:
    def treeDiameter(self, edges):
        g = build_graph(edges)
        n = len(g)
        if n <= 2:
            return n - 1
        deg = [len(nb) for nb in g]
        leaves = [v for v in range(n) if deg[v] == 1]
        remaining = n
        layers = 0
        while remaining > 2:
            layers += 1
            remaining -= len(leaves)
            nxt = []
            for leaf in leaves:
                for v in g[leaf]:
                    deg[v] -= 1
                    if deg[v] == 1:
                        nxt.append(v)
            leaves = nxt
        return 2 * layers + (1 if remaining == 2 else 0)


for edges, want in examples:
    print(SolutionPeel().treeDiameter(edges), want)

2 2
4 4
0 0


**Độ phức tạp:** mỗi đỉnh bị xoá một lần, mỗi cạnh được xét hằng số lần → `O(n)`.

Chú ý chi tiết: khi `remaining` còn 3 đỉnh dạng `a − b − c`, lá là `a`, `c`; xoá xong còn 1 đỉnh `b` → `remaining = 1` → đường kính `2 · layers`. Khi còn 4 đỉnh dạng đường thẳng, xoá 2 lá còn 2 đỉnh → cộng thêm 1.

### 7.4 Cách 5 — DFS lặp với stack (như lời giải thứ hai của doocs)

Double sweep nhưng mỗi "sweep" là DFS bằng stack `(đỉnh, cha, độ sâu)` — cũng tìm được đỉnh xa nhất vì trên cây độ sâu DFS = khoảng cách duy nhất.

In [11]:
class SolutionDFSStack:
    def treeDiameter(self, edges):
        g = build_graph(edges)

        def farthest(start):
            best, node = 0, start
            stk = [(start, -1, 0)]
            while stk:
                i, fa, t = stk.pop()
                if t > best:
                    best, node = t, i
                for j in g[i]:
                    if j != fa:
                        stk.append((j, i, t + 1))
            return node, best

        node, _ = farthest(0)
        return farthest(node)[1]


for edges, want in examples:
    print(SolutionDFSStack().treeDiameter(edges), want)

2 2
4 4
0 0


## 8. TRACE từng bước

Trace Cách 3 (tree DP) trên ví dụ 2, treo tại gốc 0. Thứ tự BFS: `0, 1, 2, 4, 3, 5`. Duyệt ngược:

| Đỉnh v | best1[v], best2[v] khi xét | Ứng viên best1+best2 | Cập nhật cha p | ans |
|---|---|---|---|---|
| 5 | 0, 0 | 0 | p=4: best1[4]=1 | 0 |
| 3 | 0, 0 | 0 | p=2: best1[2]=1 | 0 |
| 4 | 1, 0 | 1 | p=1: best1[1]=2 | 1 |
| 2 | 1, 0 | 1 | p=1: d=2, không > 2, best2[1]=2 | 1 |
| 1 | 2, 2 | **4** | p=0: best1[0]=3 | 4 |
| 0 | 3, 0 | 3 | — | 4 |

In [12]:
def trace_tree_dp(edges):
    g = build_graph(edges)
    n = len(g)
    parent = [-1] * n
    order, seen = [0], [False] * n
    seen[0] = True
    for u in order:
        for v in g[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                order.append(v)
    print("thứ tự BFS:", order)
    best1, best2 = [0] * n, [0] * n
    ans = 0
    for v in reversed(order):
        cand = best1[v] + best2[v]
        ans = max(ans, cand)
        p = parent[v]
        msg = ""
        if p >= 0:
            d = best1[v] + 1
            if d > best1[p]:
                best1[p], best2[p] = d, best1[p]
            elif d > best2[p]:
                best2[p] = d
            msg = f"cha {p}: best1={best1[p]}, best2={best2[p]}"
        print(f"v={v}: best1={best1[v]}, best2={best2[v]}, ứng viên={cand}, ans={ans} | {msg}")
    return ans


trace_tree_dp(examples[1][0])

thứ tự BFS: [0, 1, 2, 4, 3, 5]
v=5: best1=0, best2=0, ứng viên=0, ans=0 | cha 4: best1=1, best2=0
v=3: best1=0, best2=0, ứng viên=0, ans=0 | cha 2: best1=1, best2=0
v=4: best1=1, best2=0, ứng viên=1, ans=1 | cha 1: best1=2, best2=0
v=2: best1=1, best2=0, ứng viên=1, ans=1 | cha 1: best1=2, best2=2
v=1: best1=2, best2=2, ứng viên=4, ans=4 | cha 0: best1=3, best2=0
v=0: best1=3, best2=0, ứng viên=3, ans=4 | 


4

Trace Cách 4 (bóc lá) trên cùng cây:

In [13]:
def trace_peel(edges):
    g = build_graph(edges)
    n = len(g)
    deg = [len(nb) for nb in g]
    leaves = [v for v in range(n) if deg[v] == 1]
    remaining, layers = n, 0
    while remaining > 2:
        layers += 1
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            for v in g[leaf]:
                deg[v] -= 1
                if deg[v] == 1:
                    nxt.append(v)
        print(f"lớp {layers}: xoá lá {leaves}, còn {remaining} đỉnh, lá mới {nxt}")
        leaves = nxt
    ans = 2 * layers + (1 if remaining == 2 else 0)
    print("đường kính =", ans)


trace_peel(examples[1][0])

lớp 1: xoá lá [0, 3, 5], còn 3 đỉnh, lá mới [2, 4]
lớp 2: xoá lá [2, 4], còn 1 đỉnh, lá mới [1]
đường kính = 4


## 9. Code nộp LeetCode

Chọn hai lần BFS: ngắn, `O(n)`, không đệ quy.

In [14]:
from collections import deque
from typing import List


class Solution:
    def treeDiameter(self, edges: List[List[int]]) -> int:
        n = len(edges) + 1
        g = [[] for _ in range(n)]
        for a, b in edges:
            g[a].append(b)
            g[b].append(a)

        def bfs(s):
            dist = [-1] * n
            dist[s] = 0
            q = deque([s])
            last = s
            while q:
                u = q.popleft()
                last = u                      # đỉnh lấy ra cuối cùng là xa nhất
                for v in g[u]:
                    if dist[v] < 0:
                        dist[v] = dist[u] + 1
                        q.append(v)
            return last, dist[last]

        u, _ = bfs(0)
        return bfs(u)[1]


for edges, want in examples:
    assert Solution().treeDiameter(edges) == want
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


Mẹo: trong BFS, đỉnh **lấy ra cuối cùng** có khoảng cách lớn nhất, nên không cần `max` trên cả mảng.

## 10. Vì sao đúng

**Bổ đề double sweep:** Cho cây `T`, đỉnh bất kỳ `s`, và `u` là đỉnh xa `s` nhất. Khi đó `u` là đầu mút của một đường kính.

*Chứng minh:* Gọi `a − b` là một đường kính, độ dài `D`. Gọi `x` là đỉnh trên đường `a − b` **gần `s` nhất** (đường `s → x` chỉ chạm `a − b` tại `x`), nên `d(s, a) = d(s, x) + d(x, a)` và `d(s, b) = d(s, x) + d(x, b)`. Gọi `y` là đỉnh cuối cùng chung của hai đường `s → x` và `s → u`. Vì `u` xa `s` nhất: `d(s, u) ≥ d(s, a)` và `d(s, u) ≥ d(s, b)`.

- **Trường hợp `y ≠ x`** (đường tới `u` rẽ đi trước khi chạm `a − b`): `d(s, u) = d(s, y) + d(y, u) ≥ d(s, y) + d(y, x) + d(x, a)` nên `d(y, u) ≥ d(y, x) + d(x, a)`. Khi đó đường đơn `u → y → x → b` dài `d(u, y) + d(y, x) + d(x, b) ≥ 2·d(y, x) + d(x, a) + d(x, b) > D` (vì `d(y, x) > 0`) — mâu thuẫn với `D` lớn nhất. Vậy trường hợp này không xảy ra.
- **Trường hợp `y = x`** (đường `s → u` đi qua `x`): `d(s, u) = d(s, x) + d(x, u)` nên `d(x, u) ≥ max(d(x, a), d(x, b))`. Nhánh từ `x` đi tới `u` trùng hướng với **nhiều nhất một** trong hai phía `a`, `b`; chọn phía còn lại, giả sử `b`: đường đơn `u → x → b` dài `d(u, x) + d(x, b) ≥ d(x, a) + d(x, b) = D`. Vì `D` lớn nhất nên bằng `D` → `u` là đầu mút của đường kính `u − b`.

Vậy BFS lần hai từ `u` cho khoảng cách xa nhất `= D`. ∎

**Tree DP:** mỗi đường đi đơn có đúng một đỉnh gần gốc nhất `v`; đường đó gồm tối đa hai nhánh đi xuống qua **hai con khác nhau** của `v`. Độ dài lớn nhất với `v` cố định là `best1[v] + best2[v]`. Lấy `max` qua mọi `v` là xét hết mọi đường. Duyệt BFS ngược đảm bảo con được xử lý trước cha.

**Bóc lá:** hai đầu đường kính luôn là lá; xoá mọi lá làm đường kính giảm đúng 2 (đường kính mới là đường cũ bỏ hai đầu — không có đường nào khác dài hơn còn sót, vì mọi đường dài nhất trong cây mới cũng mở rộng được thêm 1 ở mỗi đầu trong cây cũ).

## 11. Time / Space Complexity

Đặt `n` = số đỉnh (`≤ 10⁴`), số cạnh `n − 1`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS từ mọi đỉnh | `O(n²)` | `O(n)` | `10⁸` với n lớn — quá chậm |
| 2. Hai lần BFS | `O(n)` | `O(n)` | Code nộp |
| 3. Tree DP lặp | `O(n)` | `O(n)` | Một lần duyệt, tổng quát hơn |
| 4. Bóc lá | `O(n)` | `O(n)` | Liên hệ bài tìm tâm cây |
| 5. DFS stack hai lần | `O(n)` | `O(n)` | Như 2, dùng stack |

Danh sách kề tốn `O(n)` bộ nhớ (`2(n − 1)` phần tử).

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [15]:
ALL = [SolutionBrute, SolutionTwoBFS, SolutionTreeDP, SolutionPeel, SolutionDFSStack, Solution]


def check(edges, want=None):
    outs = [cls().treeDiameter(edges) for cls in ALL]
    assert len(set(outs)) == 1, (edges, outs)
    if want is not None:
        assert outs[0] == want, (edges, outs[0], want)
    return outs[0]


for edges, want in examples:
    check(edges, want)
check([], 0)                                     # n = 1
check([[0, 1]], 1)                               # n = 2
check([[0, i] for i in range(1, 8)], 2)          # hình sao
check([[i, i + 1] for i in range(9)], 9)         # đường thẳng
check([[1, 0], [2, 1], [3, 1], [4, 3]], 3)       # nhãn cạnh đảo chiều
check([[0, 1], [1, 2], [2, 3], [0, 4], [4, 5], [5, 6], [6, 7]], 7)   # đường kính đi qua gốc 0
check([[0, 1], [1, 2], [1, 3], [3, 4], [3, 5], [5, 6]], 4)           # đường kính không qua gốc 0
print("edge cases OK")

edge cases OK


### 12.2 Ngẫu nhiên có seed: cây ngẫu nhiên, đối chiếu với brute force

Sinh cây ngẫu nhiên: đỉnh `i` nối với một đỉnh ngẫu nhiên `< i`, rồi **xáo nhãn** và thứ tự cạnh để không thiên vị gốc 0.

In [16]:
def random_tree(rng, n, shape="random"):
    perm = list(range(n))
    rng.shuffle(perm)
    edges = []
    for i in range(1, n):
        if shape == "path":
            p = i - 1
        elif shape == "star":
            p = 0
        else:
            p = rng.randrange(max(0, i - 3), i) if shape == "deep" else rng.randrange(i)
        a, b = perm[i], perm[p]
        edges.append([a, b] if rng.random() < 0.5 else [b, a])
    rng.shuffle(edges)
    return edges


rng = random.Random(1245)
for _ in range(500):
    n = rng.randint(1, 40)
    check(random_tree(rng, n, rng.choice(["random", "deep", "path", "star"])))
print("500 cây ngẫu nhiên OK")

500 cây ngẫu nhiên OK


### 12.3 Test giới hạn `n = 10⁴` (đường thẳng — bẫy đệ quy, và cây ngẫu nhiên)

Bỏ qua brute force ở đây (quá chậm); so các cách `O(n)` với nhau.

In [17]:
FAST = [SolutionTwoBFS, SolutionTreeDP, SolutionPeel, SolutionDFSStack, Solution]
for shape in ["path", "random", "deep", "star"]:
    edges = random_tree(random.Random(9), 10000, shape)
    res = []
    t = time.perf_counter()
    for cls in FAST:
        res.append(cls().treeDiameter(edges))
    dt = time.perf_counter() - t
    assert len(set(res)) == 1
    print(f"{shape:6}: đường kính = {res[0]:5} | 5 cách tổng {dt * 1000:.0f} ms")
assert Solution().treeDiameter(random_tree(random.Random(1), 10000, "path")) == 9999

path  : đường kính =  9999 | 5 cách tổng 29 ms


random: đường kính =    35 | 5 cách tổng 31 ms


deep  : đường kính =  5013 | 5 cách tổng 32 ms


star  : đường kính =     2 | 5 cách tổng 50 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ BFS một lần từ đỉnh 0

Nhiều bạn nghĩ "đường dài nhất xuất phát từ đỉnh 0" là đủ.

In [18]:
def wrong_one_bfs(edges):
    g = build_graph(edges)
    return max(bfs_dist(g, 0))


edges = [[0, 1], [1, 2], [2, 3], [1, 4], [4, 5]]
print("sai :", wrong_one_bfs(edges), "| đúng:", Solution().treeDiameter(edges))

sai : 3 | đúng: 4


Với ví dụ 2 của đề, BFS từ `0` cho `3` nhưng đường kính là `4`: đỉnh 0 **không** phải đầu mút đường kính. **Sửa:** BFS lần hai từ đỉnh xa nhất vừa tìm được.

### Lỗi 2 — Tree DP chỉ tính tại gốc

In [19]:
def wrong_root_only(edges):
    g = build_graph(edges)
    n = len(g)
    parent = [-1] * n
    order, seen = [0], [False] * n
    seen[0] = True
    for u in order:
        for v in g[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                order.append(v)
    best1, best2 = [0] * n, [0] * n
    for v in reversed(order):
        p = parent[v]
        if p >= 0:
            d = best1[v] + 1
            if d > best1[p]:
                best1[p], best2[p] = d, best1[p]
            elif d > best2[p]:
                best2[p] = d
    return best1[0] + best2[0]                 # SAI: chỉ xét đường qua gốc


edges = [[0, 1], [1, 2], [1, 3], [3, 4], [3, 5], [5, 6]]
print("sai :", wrong_root_only(edges), "| đúng:", Solution().treeDiameter(edges))

sai : 4 | đúng: 4


**Vì sao sai:** đường kính `2 − 1 − 3 − 5 − 6` (độ dài 4) không đi qua gốc `0`. **Sửa:** cập nhật `ans = max(ans, best1[v] + best2[v])` tại **mọi** đỉnh.

### Lỗi 3 — DFS đệ quy trên cây dạng đường thẳng 10⁴ đỉnh

In [20]:
def diameter_recursive(edges):
    g = build_graph(edges)
    ans = 0

    def dfs(u, p):
        nonlocal ans
        b1 = b2 = 0
        for v in g[u]:
            if v != p:
                d = dfs(v, u) + 1
                if d > b1:
                    b1, b2 = d, b1
                elif d > b2:
                    b2 = d
        ans = max(ans, b1 + b2)
        return b1

    dfs(0, -1)
    return ans


line = [[i, i + 1] for i in range(9999)]
try:
    diameter_recursive(line)
    print("chạy được")
except RecursionError:
    print("RecursionError: độ sâu 10^4 vượt giới hạn", sys.getrecursionlimit())
print("cây nhỏ vẫn đúng:", diameter_recursive(examples[1][0]))

RecursionError: độ sâu 10^4 vượt giới hạn 1000
cây nhỏ vẫn đúng: 4


**Vì sao sai:** thuật toán đúng nhưng Python giới hạn độ sâu đệ quy. **Sửa:** cài đặt lặp (Cách 2–5), hoặc `sys.setrecursionlimit` + chạy trong thread có stack lớn (rủi ro, không khuyến khích).

### Lỗi 4 — Tính số đỉnh thay vì số cạnh, và suy `n` từ nhãn lớn nhất

In [21]:
def wrong_count_nodes(edges):
    n = max(max(e) for e in edges) + 1          # SAI khi edges rỗng
    g = [[] for _ in range(n)]
    for a, b in edges:
        g[a].append(b)
        g[b].append(a)
    d = bfs_dist(g, 0)
    u = d.index(max(d))
    return max(bfs_dist(g, u)) + 1              # SAI: +1 đếm đỉnh


print("sai (ví dụ 1):", wrong_count_nodes(examples[0][0]), "| đúng:", Solution().treeDiameter(examples[0][0]))
try:
    wrong_count_nodes([])
except ValueError as e:
    print("edges rỗng -> ValueError:", e)

sai (ví dụ 1): 3 | đúng: 2
edges rỗng -> ValueError: max() iterable argument is empty


**Vì sao sai:** đề định nghĩa đường kính bằng **số cạnh** (`1 − 0 − 2` có 2 cạnh, 3 đỉnh). Và `max(...)` trên dãy rỗng ném lỗi khi `n = 1`. **Sửa:** `n = len(edges) + 1`, trả về khoảng cách (số cạnh).

## 14. Mẹo & nhận dạng pattern

- "Đường đi dài nhất trong cây" / "hai đỉnh xa nhau nhất" → **double sweep BFS** hoặc **tree DP hai nhánh sâu nhất**.
- Đầu vào là danh sách cạnh `n − 1` cạnh → cây vô hướng; dựng adjacency list hai chiều, duyệt với `parent` thay cho `visited`.
- `n` tới `10⁴`–`10⁵` trong Python → tránh đệ quy; dùng **thứ tự BFS ngược** để làm DP "con trước cha".
- "Bóc lá theo lớp" xuất hiện trong: 310 Minimum Height Trees (tâm cây), 2603 Collect Coins in a Tree.
- Họ hàng: 543 Diameter of Binary Tree (cây nhị phân `TreeNode`), 124 Binary Tree Maximum Path Sum (DP hai nhánh có trọng số), 1522 Diameter of N-Ary Tree, 2246 Longest Path With Different Adjacent Characters.

## 15. Biến thể

1. **Cạnh có trọng số dương:** double sweep vẫn đúng (thay BFS bằng DFS cộng trọng số). **Trọng số âm:** chỉ tree DP còn đúng (nhánh âm thì bỏ, lấy `max(0, …)`).
2. **Trả về chính đường kính (danh sách đỉnh):** BFS lần hai lưu `parent`, truy ngược từ đỉnh xa nhất.
3. **Tâm cây / chiều cao nhỏ nhất (310):** đỉnh giữa của đường kính; bóc lá cho trực tiếp.
4. **Rừng (nhiều cây):** chạy cho từng thành phần liên thông, lấy max.
5. **Đường kính sau khi nối hai cây (3203):** `max(d1, d2, ceil(d1/2) + ceil(d2/2) + 1)`.

Demo biến thể 2 và 3:

In [22]:
def diameter_path(edges):
    g = build_graph(edges)
    n = len(g)

    def bfs_parent(s):
        par = [-2] * n
        par[s] = -1
        q = deque([s])
        last = s
        while q:
            u = q.popleft()
            last = u
            for v in g[u]:
                if par[v] == -2:
                    par[v] = u
                    q.append(v)
        return last, par

    u, _ = bfs_parent(0)
    w, par = bfs_parent(u)
    path = [w]
    while par[path[-1]] != -1:
        path.append(par[path[-1]])
    return path


def tree_centers(edges):
    path = diameter_path(edges)
    L = len(path)
    return sorted({path[(L - 1) // 2], path[L // 2]})


edges = examples[1][0]
print("đường kính:", diameter_path(edges), "| tâm:", tree_centers(edges))

đường kính: [3, 2, 1, 4, 5] | tâm: [1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mọi đỉnh | max khoảng cách mọi cặp | `O(n²)` | `O(n)` | Tham chiếu kiểm thử |
| 2. Hai lần BFS | đỉnh xa nhất là đầu đường kính | `O(n)` | `O(n)` | Mặc định, dễ nhớ |
| 3. Tree DP lặp | hai nhánh sâu nhất tại mỗi đỉnh | `O(n)` | `O(n)` | Trọng số âm, cần thêm thông tin tại đỉnh |
| 4. Bóc lá | đếm số lớp lá | `O(n)` | `O(n)` | Khi đồng thời cần tâm cây |
| 5. DFS stack hai lần | như 2, dùng stack | `O(n)` | `O(n)` | Khi quen DFS |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `edges = [[0,1],[1,2],[2,3],[3,4],[2,5]]`. Đường kính?

**Bài 2 (tính tay):** với cây hình sao 6 đỉnh (tâm 0), bóc lá được mấy lớp? Còn lại bao nhiêu đỉnh?

**Bài 3 (điền TODO):** viết `farthest(g, s)` trả về `(đỉnh xa nhất, khoảng cách)` bằng BFS.

**Bài 4 (tự viết lại):** viết `diameter_weighted(n, wedges)` cho cây có trọng số dương `wedges[i] = [a, b, w]` (double sweep, dùng stack).

**Bài 5 (mini challenge):** viết `count_diameter_endpoints(edges)`: số đỉnh `v` là đầu mút của **ít nhất một** đường kính.

In [23]:
def farthest(g, s):
    # TODO
    raise NotImplementedError


def diameter_weighted(n, wedges):
    # TODO: hai lần DFS cộng trọng số
    raise NotImplementedError


def count_diameter_endpoints(edges):
    # TODO: gợi ý dùng hai đầu u, w của một đường kính
    raise NotImplementedError


# Test gợi ý:
# assert farthest(build_graph(examples[1][0]), 3) == (5, 4)
# assert diameter_weighted(3, [[0, 1, 5], [1, 2, 7]]) == 12
# assert count_diameter_endpoints([[0, 1], [0, 2], [0, 3]]) == 3

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: vẽ cây ra giấy.
- Mức 2: Bài 1: nhánh dài nhất từ đỉnh 2 là `2−3−4` (2 cạnh) và `2−1−0` (2 cạnh).
- Mức 3: Bài 2: lớp 1 xoá 5 lá, còn 1 đỉnh → đường kính `2·1 = 2`.

**Bài 3–4:**
- Mức 1: chép `bfs` trong mục 9.
- Mức 2: Bài 4: stack phần tử `(đỉnh, cha, tổng trọng số)`.
- Mức 3: hai lần: từ 0 tìm `u`, từ `u` tìm khoảng cách lớn nhất.

**Bài 5:**
- Mức 1: lấy `u`, `w` là hai đầu một đường kính độ dài `D`.
- Mức 2: tính `du = dist từ u`, `dw = dist từ w`.
- Mức 3: `v` là đầu mút một đường kính ⇔ `max(du[v], dw[v]) == D` (vì đỉnh xa nhất từ `v` luôn nằm trong `{u, w}` về khoảng cách — eccentricity `= max(du[v], dw[v])`).

## 19. Đáp án tham khảo

**Bài 1:** `4` (`0−1−2−3−4`). **Bài 2:** 1 lớp, còn 1 đỉnh, đường kính `2`.

In [24]:
assert Solution().treeDiameter([[0, 1], [1, 2], [2, 3], [3, 4], [2, 5]]) == 4
assert Solution().treeDiameter([[0, i] for i in range(1, 6)]) == 2
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
def farthest(g, s):
    dist = bfs_dist(g, s)
    u = max(range(len(g)), key=dist.__getitem__)
    return u, dist[u]


def diameter_weighted(n, wedges):
    g = [[] for _ in range(n)]
    for a, b, w in wedges:
        g[a].append((b, w))
        g[b].append((a, w))

    def far(s):
        best, node = 0, s
        stk = [(s, -1, 0)]
        while stk:
            u, p, d = stk.pop()
            if d > best:
                best, node = d, u
            for v, w in g[u]:
                if v != p:
                    stk.append((v, u, d + w))
        return node, best

    u, _ = far(0)
    return far(u)[1]


def count_diameter_endpoints(edges):
    g = build_graph(edges)
    u, _ = farthest(g, 0)
    w, D = farthest(g, u)
    du, dw = bfs_dist(g, u), bfs_dist(g, w)
    return sum(1 for v in range(len(g)) if max(du[v], dw[v]) == D)


def brute_endpoints(edges):
    g = build_graph(edges)
    all_d = [bfs_dist(g, s) for s in range(len(g))]
    D = max(map(max, all_d))
    return sum(1 for v in range(len(g)) if max(all_d[v]) == D)


assert farthest(build_graph(examples[1][0]), 3) == (5, 4)
assert diameter_weighted(3, [[0, 1, 5], [1, 2, 7]]) == 12
assert count_diameter_endpoints([[0, 1], [0, 2], [0, 3]]) == 3
rng = random.Random(55)
for _ in range(300):
    n = rng.randint(1, 30)
    edges = random_tree(rng, n)
    assert count_diameter_endpoints(edges) == brute_endpoints(edges)
    assert diameter_weighted(n, [[a, b, 1] for a, b in edges]) == Solution().treeDiameter(edges)
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (double sweep BFS):**

```python
class Solution:
    def treeDiameter(self, edges: List[List[int]]) -> int:
        n = len(edges) + 1
        g = [[] for _ in range(n)]
        for a, b in edges:
            g[a].append(b); g[b].append(a)

        def bfs(s):
            dist = [-1] * n; dist[s] = 0
            q = deque([s]); last = s
            while q:
                u = q.popleft(); last = u
                for v in g[u]:
                    if dist[v] < 0:
                        dist[v] = dist[u] + 1; q.append(v)
            return last, dist[last]

        u, _ = bfs(0)
        return bfs(u)[1]
```

**Checklist trước khi submit:**

- [ ] `n = len(edges) + 1` (xử lý `edges = []` → 0).
- [ ] Danh sách kề **hai chiều**.
- [ ] Hai lần BFS (không phải một), hoặc DP cập nhật đáp án tại **mọi** đỉnh.
- [ ] Đếm **cạnh**, không đếm đỉnh.
- [ ] Không đệ quy sâu `10⁴` (hoặc đã xử lý giới hạn đệ quy).